# Практика 1. Первый проект аналитика данных

**Что мы сделаем за пару:**

1. Создадим папку проекта и файл `requirements.txt` — список библиотек проекта.
2. Создадим виртуальное окружение и установим туда библиотеки из `requirements.txt`.
3. Сгенерируем датасет и решим задачу на **pandas**.
4. Выложим проект на **GitHub** с правильным `.gitignore`.
5. Добавим библиотеку **polars**, решим ту же задачу на ней и снова отправим код на GitHub.

**Что понадобится:** установленный Python, VS Code (или другой редактор), Git, аккаунт на github.com.

> Команды в блоках с пометкой «терминал» выполняются **в терминале**, а не в ячейке ноутбука.
> Ячейки ноутбука (прямоугольники с `[ ]` слева) запускаются сочетанием **Shift + Enter**.

---
## Шаг 0. Папка проекта

1. Создайте у себя папку `sales-analytics` (**без пробелов и русских букв в названии**).
2. Откройте её в VS Code: `Файл → Открыть папку...`
3. Откройте встроенный терминал: `Терминал → Новый терминал`.
4. Положите этот ноутбук внутрь папки проекта.

Проверьте, что терминал «стоит» в нужной папке:

```
# терминал
pwd     # macOS / Linux
cd      # Windows PowerShell — покажет текущий путь
```

---
## Шаг 1. Файл `requirements.txt`

`requirements.txt` — это **список библиотек, которые нужны проекту**. Одна строка = одна библиотека.

Зачем он нужен: любой человек (преподаватель, коллега и вы сами через полгода) сможет одной командой
поставить ровно те же библиотеки и запустить ваш код. Без этого файла проект «работает только у меня на ноутбуке».

### Задание
Создайте в корне проекта файл `requirements.txt` и напишите в нём одну строку:

```
pandas
```

Создать файл можно мышкой в VS Code (правой кнопкой по папке → New File) или командой:

```
# терминал (Windows PowerShell)
"pandas" | Out-File -Encoding utf8 requirements.txt
```

```
# терминал (macOS / Linux)
echo "pandas" > requirements.txt
```

---
## Шаг 2. Виртуальное окружение (venv)

**Проблема:** библиотеки, установленные «глобально», общие для всех проектов. Одному проекту нужна
старая версия pandas, другому — новая, и они конфликтуют.

**Решение:** для каждого проекта — своя отдельная папка с Python и библиотеками. Это и есть
виртуальное окружение (virtual environment, venv). Аналогия: отдельный рюкзак с инструментами под каждый предмет.

### 2.1 Создать окружение

```
# терминал (Windows)
python -m venv .venv
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
```

Появится папка `.venv` — руками её содержимое трогать не нужно.

### 2.2 Активировать окружение

```
# терминал (Windows PowerShell)
.\.venv\Scripts\Activate.ps1
```

```
# терминал (macOS / Linux)
source .venv/bin/activate
```

Признак успеха: в начале строки терминала появилось `(.venv)`.

### 2.3 Установить библиотеки из requirements.txt

```
# терминал
pip install -r requirements.txt
pip list
```

### 2.4 Подключить окружение к ноутбуку

В VS Code справа сверху нажмите **Select Kernel** → **Python Environments** → выберите `.venv`.
Если предложит установить `ipykernel` — соглашайтесь.

### Проверка: то ли окружение подключено?

Запустите ячейку ниже. В пути к Python должно встретиться `.venv`.

In [ ]:
import sys

print("Python отсюда:", sys.executable)
print("Версия Python:", sys.version.split()[0])

import pandas as pd
print("pandas версии:", pd.__version__)

---
## Шаг 3. Данные

Запустите ячейку ниже — она создаст файл `data/companies.csv` с данными о выручке компаний.
Менять в ней ничего не нужно.

**Колонки датасета:**

| Колонка | Что означает |
|---|---|
| `company` | название компании |
| `industry` | отрасль |
| `city` | город |
| `year` | год (2022, 2023, 2024) |
| `revenue_mln` | выручка за год, млн руб. |
| `employees` | число сотрудников |

In [ ]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Просто запустите её (Shift+Enter), менять ничего не нужно.
# Она создаст папку data/ и файл data/companies.csv

import csv
import random
from pathlib import Path

random.seed(42)  # фиксируем случайность, чтобы у всех получился одинаковый датасет

industries = ["IT", "Ритейл", "Транспорт", "Энергетика",
              "Финансы", "Строительство", "Пищепром", "Телеком"]
cities = ["Москва", "Санкт-Петербург", "Казань", "Новосибирск",
          "Екатеринбург", "Нижний Новгород", "Сочи", "Владивосток"]
first = ["Альфа", "Вектор", "Гранит", "Дельта", "Заря", "Импульс", "Кристалл",
         "Лидер", "Меридиан", "Нова", "Орион", "Прогресс", "Ритм", "Сигма",
         "Титан", "Униор", "Феникс", "Хорда", "Центавр", "Эверест"]
second = ["Групп", "Логистик", "Систем", "Инвест", "Технолоджис", "Трейд", "Пром", "Софт"]

all_names = [f"{a}-{b}" for a in first for b in second]
companies = random.sample(all_names, 60)

rows = []
for name in companies:
    industry = random.choice(industries)
    city = random.choice(cities)
    base_revenue = round(random.uniform(50, 4000), 1)      # выручка в млн руб.
    people_per_mln = random.uniform(0.08, 0.5)
    for year in (2022, 2023, 2024):
        growth = random.uniform(0.85, 1.35) ** (year - 2022)
        revenue = round(base_revenue * growth, 1)
        employees = max(5, int(revenue * people_per_mln))
        rows.append({
            "company": name,
            "industry": industry,
            "city": city,
            "year": year,
            "revenue_mln": revenue,
            "employees": employees,
        })

Path("data").mkdir(exist_ok=True)
with open("data/companies.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Готово! Создан файл data/companies.csv")
print("Строк в файле:", len(rows))
print("Колонки:", list(rows[0].keys()))
print("Первая строка:", rows[0])

---
## Шаг 4. Задание на pandas

### 4.1 Загрузите данные и посмотрите на них

Подсказки: `pd.read_csv("data/companies.csv")`, метод `.head()`, атрибут `.shape`.

In [ ]:
import pandas as pd

# 1) прочитайте файл data/companies.csv в переменную df
df = ...

# 2) выведите первые 5 строк


# 3) выведите размер таблицы (сколько строк и колонок)

### 4.2 Основная задача

**Выведите топ-10 компаний по выручке за 2024 год.**
В результате должны остаться только колонки `company`, `industry`, `revenue_mln`.

План решения:
1. оставить только строки, где `year` равен 2024 — это **фильтрация**;
2. отсортировать по `revenue_mln` по убыванию — метод `.sort_values(...)` с аргументом `ascending=False`;
3. взять первые 10 строк — метод `.head(10)`;
4. оставить нужные колонки — `df[["company", "industry", "revenue_mln"]]`.

In [ ]:
# Задание: топ-10 компаний по выручке за 2024 год (pandas)

top10_pandas = ...

top10_pandas

### 4.3 Дополнительно (если успеваете)

1. Сколько всего **уникальных** компаний в датасете? (подсказка: `.nunique()`)
2. Какая **средняя выручка по каждой отрасли** в 2024 году? Отсортируйте по убыванию.
   (подсказка: `.groupby("industry")["revenue_mln"].mean()`)
3. Сколько строк в датасете относится к Москве?

In [ ]:
# место для дополнительных заданий

---
## Шаг 5. Проект на GitHub

### 5.1 Файл `.gitignore`

В git **не должно попадать** то, что легко пересоздать или что много весит: в первую очередь
папка виртуального окружения `.venv` (это тысячи файлов и сотни мегабайт). Именно поэтому мы и
храним `requirements.txt` — по нему окружение восстанавливается одной командой.

Создайте в корне проекта файл `.gitignore` (имя начинается с точки!) со следующим содержимым:

```
.venv/
venv/
__pycache__/
.ipynb_checkpoints/
.DS_Store
.idea/
```

### 5.2 Первый коммит

```
# терминал
git init
git add .
git status
```

В выводе `git status` **убедитесь, что папки `.venv` там нет.** Если она есть — проверьте `.gitignore`.

Если git не знает, кто вы (это бывает только при самом первом использовании):

```
# терминал
git config --global user.name "Ваше Имя"
git config --global user.email "ваша@почта"
```

Делаем коммит:

```
# терминал
git commit -m "Проект аналитики: requirements, данные, решение на pandas"
```

### 5.3 Репозиторий на GitHub

1. Зайдите на github.com → кнопка **New repository**.
2. Имя: `sales-analytics`, видимость **Public**.
3. **Не ставьте** галочки «Add README», «Add .gitignore» — репозиторий должен быть пустым.
4. Скопируйте адрес репозитория и выполните:

```
# терминал (подставьте свою ссылку)
git branch -M main
git remote add origin https://github.com/ВАШ_ЛОГИН/sales-analytics.git
git push -u origin main
```

Обновите страницу репозитория — ваши файлы должны быть на месте.

---
## Шаг 6. Библиотека polars

**polars** — более новая библиотека для работы с таблицами. Делает то же самое, что pandas,
но обычно быстрее и с другим синтаксисом. Наша цель — увидеть, что инструмент можно поменять,
а логика решения задачи остаётся той же.

### 6.1 Добавьте polars в проект

Допишите в `requirements.txt` вторую строку, чтобы файл стал таким:

```
pandas
polars
```

И установите:

```
# терминал (окружение должно быть активно, в строке видно (.venv))
pip install -r requirements.txt
```

> Важно: сначала правим `requirements.txt`, потом устанавливаем. Так список библиотек всегда
> соответствует реальному составу проекта.

### 6.2 Та же задача на polars

**Выведите топ-10 компаний по выручке за 2024 год**, используя polars.

Подсказки по синтаксису:

| Действие | pandas | polars |
|---|---|---|
| прочитать csv | `pd.read_csv(path)` | `pl.read_csv(path)` |
| фильтр | `df[df["year"] == 2024]` | `df.filter(pl.col("year") == 2024)` |
| сортировка | `df.sort_values("revenue_mln", ascending=False)` | `df.sort("revenue_mln", descending=True)` |
| первые 10 строк | `df.head(10)` | `df.head(10)` |
| выбрать колонки | `df[["a", "b"]]` | `df.select(["a", "b"])` |

В polars шаги удобно соединять в цепочку через точку.

In [ ]:
import polars as pl

# 1) прочитайте data/companies.csv в переменную df_pl
df_pl = ...

# 2) топ-10 компаний по выручке за 2024 год
top10_polars = ...

top10_polars

### 6.3 Сравните результаты

Совпали ли компании в ответах pandas и polars? Так и должно быть: библиотеки разные, данные и ответ — одинаковые.

In [ ]:
# выведите оба результата и сравните

---
## Шаг 7. Отправляем обновления на GitHub

```
# терминал
git status
git add .
git commit -m "Добавлен polars и решение той же задачи на polars"
git push
```

Проверьте на github.com, что появился второй коммит и обновлённый `requirements.txt`.

---
## Чек-лист сдачи

- [ ] В репозитории есть `requirements.txt` с двумя строками: `pandas` и `polars`
- [ ] В репозитории есть `.gitignore`, и папки `.venv` в репозитории **нет**
- [ ] В репозитории лежит этот ноутбук с решёнными заданиями
- [ ] Задача «топ-10 компаний по выручке за 2024» решена на pandas
- [ ] Та же задача решена на polars
- [ ] Сделано минимум два коммита
- [ ] Ссылка на репозиторий отправлена преподавателю

**Что важно унести с пары:** проект — это не только файл с кодом. Это код + список зависимостей +
изолированное окружение + история изменений в git. Именно так выглядит работа аналитика в команде.